# first get book names, shortnames for scrapin

In [ ]:
books_ot_str ="""
"Gen" "Exo" "Lev" "Num" "Deu" "Jos" "Jdg" "Rth" "1Sa" "2Sa" "1Ki" "2Ki" "1Ch" "2Ch" "Ezr" "Neh" "Est" "Job" "Psa" "Pro" "Ecc" "Sng" "Isa" "Jer" "Lam" "Eze" "Dan" "Hos" "Joe" "Amo" "Oba" "Jon" "Mic" "Nah" "Hab" "Zep" "Hag" "Zec" "Mal"
"""
bkslist = ["genesis", "exodus", "leviticus", "numbers", "deuteronomy", "joshua", "judges", "ruth", "1_samuel", "2_samuel", "1_kings", "2_kings", "1_chronicles", "2_chronicles", "ezra", "nehemiah", "esther", "job", "psalms", "proverbs", "ecclesiastes", "songs", "isaiah", "jeremiah", "lamentations", "ezekiel", "daniel", "hosea", "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk", "zephaniah", "haggai", "zechariah", "malachi", "matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]
chaps_ot_str = """
50 40 27 36 34 24 21 4 31 24 22 25 29 36 10 13 10 42 150 31 12 8 66 52 5 48 12 14 3 9 1 4 7 3 3 3 2 14 4
"""

In [ ]:
books_ot = books_ot_str.strip().strip('"').split("\" \"")

In [ ]:
chaps_ot = [int(c) for c in chaps_ot_str.strip().split(" ")]

In [ ]:
#def blb_to_bks(name):
#    s = name.lower()
#    if s[0].is_numeric():
#        s = s[0]+"_"+s[1:]
# all good 1:1 order :))
#for i in range (len(books_ot)):
#    print(books_ot[i].lower()[:3], bkslist[i].lower()[:3] if not bkslist[i][0].isnumeric() else bkslist[i].lower()[:4])

# parsee

In [ ]:
POS_MAP = {
    "V": "Verb",
    "N": "Noun",
    "Adv": "Adverb",
    "Adj": "Adjective",
    "Art": "Article",
    "DPro": "Demonstrative Pronoun",
    "IPro": "Interrogative / Indefinite Pronoun",
    "PPro": "Personal / Possessive Pronoun",
    "RecPro": "Reciprocal Pronoun",
    "RelPro": "Relative Pronoun",
    "RefPro": "Reflexive Pronoun",
    "Prep": "Preposition",
    "Conj": "Conjunction",
    "I": "Interjection",
    "Prtcl": "Particle",
    "PrtclDsj": "Particle, Disjunctive Particle",
    "Heb": "Hebrew Word",
    "Aram": "Aramaic Word"
}

PERSON_MAP = {"1": "1st Person", "2": "2nd Person", "3": "3rd Person"}
TENSE_MAP = {"P": "Present", "I": "Imperfect", "F": "Future",
             "A": "Aorist", "R": "Perfect", "L": "Pluperfect"}
MOOD_MAP = {"I": "Indicative", "M": "Imperative",
            "S": "Subjunctive", "O": "Optative",
            "N": "Infinitive", "P": "Participle"}
VOICE_MAP = {"A": "Active", "M": "Middle", "P": "Passive", "M/P": "Middle or Passive"}
CASE_MAP = {"N": "Nominative", "V": "Vocative", "A": "Accusative",
            "G": "Genitive", "D": "Dative"}
NUMBER_MAP = {"S": "Singular", "P": "Plural"}
GENDER_MAP = {"M": "Masculine", "F": "Feminine", "N": "Neuter"}
COMPARISON_MAP = {"C": "Comparative", "S": "Superlative"}

In [ ]:
import re
from bs4 import BeautifulSoup

# Izveidojam apgrieztās vārdnīcas (Reverse Maps)
def reverse_map(d): return {v: k for k, v in d.items()}

R_POS = reverse_map(POS_MAP)
R_POS["Definite article"] = "Art"  # BLB specifika
R_PERSON = reverse_map(PERSON_MAP)
R_TENSE = reverse_map(TENSE_MAP)
R_MOOD = reverse_map(MOOD_MAP)
R_VOICE = reverse_map(VOICE_MAP)
R_CASE = reverse_map(CASE_MAP)
R_NUMBER = reverse_map(NUMBER_MAP)
R_GENDER = reverse_map(GENDER_MAP)

import re
import csv
from bs4 import BeautifulSoup

def get_morph_code(word_span):
    """Pārvērš BlueLetterBible pilnos nosaukumus uz tavu kodēto morph formātu"""
    attrs = word_span.attrs
    pos_full = attrs.get('data-speech', '')
    pos = R_POS.get(pos_full, pos_full)
    
    # Ja POS nav starp deklinējamiem/konjugējamiem, atdodam tikai POS (Prep, Conj, etc.)
    if pos not in ["V", "N", "Adj", "Art"] and "Pro" not in pos:
        return pos

    # VERBU loģika: V-TenseVoiceMood-PersonNumber vai V-TVM-CaseNumberGender
    if pos == "V":
        t = R_TENSE.get(attrs.get('data-tense', ''), '.')
        v = R_VOICE.get(attrs.get('data-voice', ''), '.')
        m = R_MOOD.get(attrs.get('data-mood', ''), '.')
        tmv = f"{t}{v}{m}"
        
        if m == "P": # Participle
            c = R_CASE.get(attrs.get('data-case', ''), '.')
            n = R_NUMBER.get(attrs.get('data-number', ''), '.')
            g = R_GENDER.get(attrs.get('data-gender', ''), '.')
            return f"V-{tmv}-{c}{n}{g}"
        else: # Finite verb
            p = R_PERSON.get(attrs.get('data-person', ''), '.')
            n = R_NUMBER.get(attrs.get('data-number', ''), '.')
            return f"V-{tmv}-{p}{n}"

    # NOMINĀLO vārdu loģika: POS-CaseNumberGender
    c = R_CASE.get(attrs.get('data-case', ''), '')
    n = R_NUMBER.get(attrs.get('data-number', ''), '')
    g = R_GENDER.get(attrs.get('data-gender', ''), '')
    return f"{pos}-{c}{n}{g}"

import os
def file_path_to_book_chapter(file_path):
    s, cc = file_path.split(os.sep)[-1].split(".")[0].split("_")
    return bkslist[books_ot.index(s)], int(cc)

def extract_to_flat_list(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    b, c = file_path_to_book_chapter(file_path)
    all_rows = []
    
    # Ejam cauri katram panta konteineram, lai saglabātu numerāciju
    verse_containers = soup.find_all('div', class_='scriptureText')
    
    for vn, container in enumerate(verse_containers):
        # Iegūstam panta numuru (piemēram, "1", "2"...)
        #TODO: extract and verify
        #vn = int(verse_num[:-2])
        verse_num = int(container.get('data-print-verse-prefix'))
        #print(verse_num)
        #print(int(verse_num[:-2]))
        
        # Atrodam visus vārdus ŠAJĀ pantā
        word_spans = container.find_all('span', class_='word-phrase')
        
        for wn, span in enumerate(word_spans):
            # 1. Tīrs grieķu teksts
            greek = "".join([t for t in span.contents if isinstance(t, str)]).strip()
            if not greek: continue
            
            # 2. Strong numurs (tikai cipari)
            strong = re.sub(r'\D', '', span.get('data-strongs', ''))
            
            # 3. Morph kods (mūsu ETL transformācija)
            morph = get_morph_code(span)
            
            all_rows.append({
                "verse": verse_num,
                "word": wn,
                "form": greek,
                "strong_num": strong,
                "strong_en_title": morph,
                "book": b,
                "chapter": c
            })
            
    return all_rows

import csv

def save_to_csv(data, output_filename):
    keys = ["verse", "word", "form", "strong_num", "strong_en_title", "book", "chapter"]
    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(data)

In [ ]:
results = extract_to_flat_list("_scrap/Gen_33.html")
for r in results[20:35]: print(f"{r['form']} | {r['strong_num']} | {r['strong_en_title']}")

In [ ]:
import time
from datetime import timedelta

# ... (iepriekšējās extract_to_flat_list un save_to_csv funkcijas) ...

results = []
tot_bks = len(books_ot)
global_start_time = time.time()

print(f"{'Nr':<3} | {'Grāmata':<10} | {'Nodaļas':<7} | {'Laiks (gr.)':<12} | {'Kopā pagājis'}")
print("-" * 65)

for cbk, (b, c) in enumerate(zip(books_ot, chaps_ot), 1):
    book_start_time = time.time()
    
    # Apstrādājam visas nodaļas
    for i in range(1, c + 1):
        file_path = f"_scrap/{b}_{i}.html"
        try:
            results.extend(extract_to_flat_list(file_path))
        except FileNotFoundError:
            print(f"Brīdinājums: Fails {file_path} netika atrasts.")

    # Aprēķinām laiku
    book_elapsed = time.time() - book_start_time
    total_elapsed = time.time() - global_start_time
    
    # Izvade terminālī
    # timedelta palīdz smuki noformēt sekundes uz HH:MM:SS
    print(f"{cbk:<3} | {b:<10} | {c:<7} | {book_elapsed:>10.2f}s | {str(timedelta(seconds=int(total_elapsed)))}")

# Saglabājam rezultātu
save_to_csv(results, "greek-lxx-ot-blb.csv")

final_time = time.time() - global_start_time
print("-" * 65)
print(f"Pabeigts! Kopējais laiks: {str(timedelta(seconds=int(final_time)))}")
print(f"Kopā apstrādāti {len(results)} vārdi.")